In [2]:
%pip install pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 22.3 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 67.0 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/2 [numpy]  WARNING: The scripts f2py and numpy-config are installed in '/usr/local/python/3.12.1/bin' which is not on PATH.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]2m1/2 [pandas]

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd

orders = pd.read_csv("orders.csv")

print("Dataset size:", orders.shape)
display(orders.head())

Dataset size: (59386, 15)


,order_id,merchant_id,route_id,processor,created_at,completed_at,order_type,status,amount,currency,amount_usd,customer_id,customer_country,payment_method,decline_reason
0,5100001,NOVA-FX,KRTX-03,KRTX,2026-04-01 16:38:16,2026-04-01 16:39:23,SALE,APPROVED,2320.41,PHP,40.60,CU-106677,PH,CARD,NaN
1,5100002,NOVA-FX,KRTX-03,KRTX,2026-04-01 21:26:47,2026-04-01 21:27:35,SALE,APPROVED,2127708.00,JPY,NaN,CU-848923,JP,CARD,NaN
2,5100003,NOVA-FX,NaN,NaN,2026-04-01 00:17:28,2026-04-01 00:18:03,SALE,FILTERED,14135.00,JPY,90.10,CU-978938,JP,CARD,blocked_pre_routing
3,5100004,NOVA-FX,SORVA-14,SORVA,2026-04-01 22:16:46,2026-04-01 22:18:42,SALE,APPROVED,18598.00,JPY,118.42,CU-768122,JP,CARD,NaN
4,5100005,NOVA-FX,SORVA-22,SORVA,2026-04-01 08:52:30,2026-04-01 08:53:41,SALE,approved,20676.33,PHP,363.40,CU-391357,PH,EWALLET,NaN


## 1. Load and understand the orders data

The orders dataset contains one row for each payment attempt. I will first inspect its size, columns, data types, sample records, and date range before applying any filters or calculations.

In [4]:
import pandas as pd

pd.set_option("display.max_columns", None)

In [5]:
# Load the main orders dataset
orders = pd.read_csv("orders.csv")

print("Dataset size:", orders.shape)
print("\nColumns:")
print(orders.columns.tolist())

print("\nFirst five rows:")
display(orders.head())

print("\nMissing values:")
display(orders.isna().sum().sort_values(ascending=False))

print("\nDuplicate rows:", orders.duplicated().sum())
print("Duplicate order IDs:", orders["order_id"].duplicated().sum())

print("\nDate range:")
print("From:", orders["created_at"].min())
print("To:", orders["created_at"].max())

Dataset size: (59386, 15)

Columns:
['order_id', 'merchant_id', 'route_id', 'processor', 'created_at', 'completed_at', 'order_type', 'status', 'amount', 'currency', 'amount_usd', 'customer_id', 'customer_country', 'payment_method', 'decline_reason']

First five rows:


,order_id,merchant_id,route_id,processor,created_at,completed_at,order_type,status,amount,currency,amount_usd,customer_id,customer_country,payment_method,decline_reason
0,5100001,NOVA-FX,KRTX-03,KRTX,2026-04-01 16:38:16,2026-04-01 16:39:23,SALE,APPROVED,2320.41,PHP,40.60,CU-106677,PH,CARD,NaN
1,5100002,NOVA-FX,KRTX-03,KRTX,2026-04-01 21:26:47,2026-04-01 21:27:35,SALE,APPROVED,2127708.00,JPY,NaN,CU-848923,JP,CARD,NaN
2,5100003,NOVA-FX,NaN,NaN,2026-04-01 00:17:28,2026-04-01 00:18:03,SALE,FILTERED,14135.00,JPY,90.10,CU-978938,JP,CARD,blocked_pre_routing
3,5100004,NOVA-FX,SORVA-14,SORVA,2026-04-01 22:16:46,2026-04-01 22:18:42,SALE,APPROVED,18598.00,JPY,118.42,CU-768122,JP,CARD,NaN
4,5100005,NOVA-FX,SORVA-22,SORVA,2026-04-01 08:52:30,2026-04-01 08:53:41,SALE,approved,20676.33,PHP,363.40,CU-391357,PH,EWALLET,NaN



Missing values:


decline_reason      40471
amount_usd           9203
route_id             5435
processor            5435
completed_at         1298
customer_country      724
merchant_id             0
order_type              0
created_at              0
order_id                0
status                  0
currency                0
amount                  0
customer_id             0
payment_method          0
dtype: int64


Duplicate rows: 0
Duplicate order IDs: 0

Date range:
From: 2026-04-01 00:13:05
To: 2026-06-30 23:59:26


In [6]:
import pandas as pd
import numpy as np
import json

# -------------------------
# 1. Load data
# -------------------------
orders = pd.read_csv(
    "orders.csv",
    parse_dates=["created_at", "completed_at"]
)

dashboard = pd.read_csv(
    "merchant_dashboard_export.csv",
    parse_dates=["date"]
)

print("Orders shape:", orders.shape)
print("Duplicate rows:", orders.duplicated().sum())
print("Duplicate order IDs:", orders["order_id"].duplicated().sum())

# -------------------------
# 2. Clean inconsistent text
# -------------------------
text_columns = [
    "merchant_id", "route_id", "processor", "order_type",
    "status", "currency", "customer_country",
    "payment_method", "decline_reason"
]

for column in text_columns:
    orders[column] = (
        orders[column]
        .astype("string")
        .str.strip()
        .str.upper()
    )

# -------------------------
# 3. Select Nova deposits
# -------------------------
nova = orders[
    (orders["merchant_id"] == "NOVA-FX") &
    (orders["order_type"] == "SALE")
].copy()

# Nova's dashboard uses UTC+8
nova["local_date"] = (
    nova["created_at"] + pd.Timedelta(hours=8)
).dt.normalize()

# Exclude June 29–30 because the extract is incomplete
nova = nova[nova["local_date"] <= "2026-06-28"]

# Provider approval rate uses only final outcomes
final = nova[
    nova["status"].isin(["APPROVED", "DECLINED"])
].copy()

# -------------------------
# 4. Compare May and June
# -------------------------
may = final[
    (final["local_date"] >= "2026-05-01") &
    (final["local_date"] < "2026-06-01")
]

june = final[
    (final["local_date"] >= "2026-06-01") &
    (final["local_date"] <= "2026-06-28")
]

def approval_summary(data):
    attempts = len(data)
    approved = data["status"].eq("APPROVED").sum()

    return pd.Series({
        "attempts": attempts,
        "approved": approved,
        "approval_rate": approved / attempts
    })

period_summary = pd.DataFrame({
    "May": approval_summary(may),
    "June 1-28": approval_summary(june)
}).T

display(period_summary.style.format({
    "approval_rate": "{:.1%}"
}))

# -------------------------
# 5. June performance by route
# -------------------------
route_summary = (
    june.groupby("route_id")
    .agg(
        attempts=("order_id", "size"),
        approved=("status", lambda x: x.eq("APPROVED").sum())
    )
)

route_summary["approval_rate"] = (
    route_summary["approved"] / route_summary["attempts"]
)

display(
    route_summary.sort_values("approval_rate")
    .style.format({"approval_rate": "{:.1%}"})
)

# -------------------------
# 6. Existing routes vs new NBLX route
# -------------------------
new_route = june[june["route_id"] == "NBLX-07"]
existing_routes = june[june["route_id"] != "NBLX-07"]

print(
    "Existing routes June approval rate:",
    f"{existing_routes['status'].eq('APPROVED').mean():.1%}"
)

print(
    "NBLX-07 June approval rate:",
    f"{new_route['status'].eq('APPROVED').mean():.1%}"
)

# -------------------------
# 7. Investigate Japanese SORVA cards
# -------------------------
sorva_jpy_may = may[
    (may["route_id"] == "SORVA-14") &
    (may["currency"] == "JPY")
]

sorva_jpy_after = june[
    (june["route_id"] == "SORVA-14") &
    (june["currency"] == "JPY") &
    (june["local_date"] >= "2026-06-11")
]

may_sorva_rate = sorva_jpy_may["status"].eq("APPROVED").mean()
after_sorva_rate = sorva_jpy_after["status"].eq("APPROVED").mean()

estimated_lost_approvals = (
    len(sorva_jpy_after) * may_sorva_rate
    - sorva_jpy_after["status"].eq("APPROVED").sum()
)

print("\nSORVA-14 Japanese cards")
print("May rate:", f"{may_sorva_rate:.1%}")
print("June 11-28 rate:", f"{after_sorva_rate:.1%}")
print("Estimated lost approvals:", round(estimated_lost_approvals))

# -------------------------
# 8. Risk-filtered NBLX attempts
# -------------------------
nblx_all = nova[
    (nova["route_id"] == "NBLX-07") &
    (nova["local_date"] >= "2026-06-01")
]

filtered_nblx = nblx_all["status"].eq("FILTERED").sum()

decided_nblx = nblx_all[
    nblx_all["status"].isin(
        ["APPROVED", "DECLINED", "FILTERED"]
    )
]

end_to_end_rate = (
    decided_nblx["status"].eq("APPROVED").mean()
)

print("\nNBLX filtered attempts:", filtered_nblx)
print(
    "NBLX approval including filtered attempts:",
    f"{end_to_end_rate:.1%}"
)

Orders shape: (59386, 15)
Duplicate rows: 0
Duplicate order IDs: 0


,attempts,approved,approval_rate
May,6240.000000,4996.000000,80.1%
June 1-28,8156.000000,5904.000000,72.4%


,attempts,approved,approval_rate
route_id,,,
NBLX-07,2182,1171,53.7%
KRTX-11,1015,713,70.2%
SORVA-14,1524,1182,77.6%
SORVA-22,754,585,77.6%
KRTX-03,2681,2253,84.0%


Existing routes June approval rate: 79.2%
NBLX-07 June approval rate: 53.7%

SORVA-14 Japanese cards
May rate: 74.7%
June 11-28 rate: 56.1%
Estimated lost approvals: 64

NBLX filtered attempts: 655
NBLX approval including filtered attempts: 41.3%
